# GroundFormer ORIGINAL — chấm TRAIN / VAL / TEST cuối mỗi epoch + W&B

**Bật GPU, Run all. 15 epoch từ đầu.** Giữ model, CLIP-L, optimizer, objective và decoder GitHub. Cuối **mỗi epoch**, chấm toàn bộ **34.132 dòng train / 3.358 val / 5.553 test** ở chế độ inference và ghi lên W&B theo epoch. Log minibatch nằm ở `optim/*`, tách với `train/*` ở cuối epoch.

QA/VQA có cho cả ba split. Val/test có IoP@0.3/0.5, mIoP, IoU@0.3/0.5, mIoU, GQA, pointRate và QA-đúng-ground-sai. **Train không có nhãn thời gian GT**, nên IoP/IoU/GQA train được ghi `N/A` (JSON null / W&B NaN + `grounding_available=0`), không bịa bằng 0.

Best checkpoint chỉ chọn bằng **QA validation cao nhất**. Test được quan sát mỗi epoch đúng theo yêu cầu; nó không được dùng tự động để chọn model. Kết quả best-val trên test lưu riêng ở `selected_test/*`. Bản này train mới, không lấy trọng số C1 cũ.

W&B online cần `WANDB_API_KEY` trong Colab Secrets hoặc phiên W&B đã đăng nhập. Không ghi API key vào notebook. Chấm toàn bộ train ở cuối epoch làm tăng thời gian so với notebook chỉ chấm val.


In [ ]:
# GroundFormer gốc, seed/batch/loss/architecture giữ như script GitHub.
CONFIG = {
    'run_id': 'nextgqa_bs16_original_15ep_every_epoch_tvt_v1',
    'epochs': 15,  # GitHub mặc định 30; đây là ngân sách 15 epoch người dùng chọn.
    'repo_dir': '/content/GroundFormer_Original_96b',
    'save_root': '/content/drive/MyDrive/GroundFormer/checkpoints',
    'wandb_project': 'GroundFormer-NExTGQA',
    'wandb_mode': 'online',
    'download_zip': True,
}
assert CONFIG['epochs']==15
assert CONFIG['wandb_mode'] in ['online','offline','disabled']


In [ ]:
# 1) Kiểm tra GPU
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Colab.")

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {vram:.1f} GB")


In [ ]:
from pathlib import Path
import subprocess
ROOT=Path(CONFIG['repo_dir'])
PINNED_REV='96b7e92040d8ef65d240ae3c2d135134ae9c7225'
if not ROOT.exists():
    subprocess.check_call(['git','clone','https://github.com/jinhseo/GroundFormer.git',str(ROOT)])
    subprocess.check_call(['git','-C',str(ROOT),'checkout','--detach',PINNED_REV])
elif not (ROOT/'.git').exists():raise RuntimeError('repo_dir đã tồn tại nhưng không phải checkout được quản lý')
commit=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
if commit!=PINNED_REV:raise ValueError('Checkout sai commit; chọn repo_dir mới, không tự reset thư mục hiện có')
print('GroundFormer commit:',commit)


In [ ]:
# 3) Dependencies — không downgrade PyTorch của Colab
import sys
import subprocess

packages = [
    "transformers==4.57.1",
    "h5py",
    "pandas",
    "Pillow",
    "PyYAML",
    "scikit-learn",
    "scipy",
    "sentencepiece",
    "tqdm",
    "gdown",
    "wandb==0.22.3",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir",
    *packages,
])

import torch
import transformers
import h5py
import wandb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("h5py:", h5py.__version__)
print("wandb:", wandb.__version__)


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
if CONFIG['wandb_mode']=='online':
    key=os.environ.get('WANDB_API_KEY')
    if not key:
        try:
            from google.colab import userdata
            key=userdata.get('WANDB_API_KEY')
        except Exception:pass
    if not key:
        try:key=wandb.api.api_key
        except Exception:pass
    if not isinstance(key,str) or not key.strip():
        raise RuntimeError('Thêm WANDB_API_KEY vào Colab Secrets và bật quyền truy cập notebook. Chưa train; không ghi key vào CONFIG.')
    os.environ['WANDB_API_KEY']=key.strip();wandb.login(key=key.strip())
os.environ['WANDB_MODE']=CONFIG['wandb_mode']
print('W&B mode:',CONFIG['wandb_mode'])


In [ ]:
# 5) Dùng NExT-GQA files có sẵn trong GroundFormer repo
from pathlib import Path

ROOT = Path(CONFIG["repo_dir"])
ANNO_DIR = ROOT / "datasets" / "nextgqa"

required_repo_data = [
    ANNO_DIR / "train.csv",
    ANNO_DIR / "val.csv",
    ANNO_DIR / "test.csv",
    ANNO_DIR / "gsub_val.json",
    ANNO_DIR / "gsub_test.json",
    ANNO_DIR / "frame2time_val.json",
    ANNO_DIR / "frame2time_test.json",
    ANNO_DIR / "vocab.json",
]

missing = [p for p in required_repo_data if not p.exists()]
if missing:
    raise RuntimeError(
        "GroundFormer repo đang thiếu dataset files:\n"
        + "\n".join(str(p) for p in missing)
    )

print("✅ NExT-GQA files trong repo:")
for p in required_repo_data:
    print(" ", p.relative_to(ROOT))


In [ ]:
# Dùng lại feature CLIP-L gốc trên Drive; không trích lại video.
from pathlib import Path
import shutil, zipfile
data_dir=Path('/content/data/nextgqa/CLIPL');data_dir.mkdir(parents=True,exist_ok=True)
feature_roots=[Path('/content/drive/MyDrive/QEST/data/features/CLIPL/CLIPL'),
               Path('/content/drive/MyDrive/QEST/data/features/CLIPL')]
required_features=[data_dir/f'CLIPL_I_{split}.h5' for split in ['train','val','test']]
for destination in required_features:
    if destination.exists():continue
    source=next((r/destination.name for r in feature_roots if (r/destination.name).is_file()),None)
    if source:
        print('Copy cache gốc:',source.name);shutil.copy2(source,destination)
if not all(p.is_file() for p in required_features):
    import gdown
    archive=data_dir/'CLIPL_original.zip'
    if not archive.exists():
        downloaded=gdown.download(id='101W4r6ibXJE2IOr6MINbNIMC3MFiN-us',output=str(archive),quiet=False)
        if not downloaded:raise RuntimeError('Không tải được cache CLIP-L gốc')
    with zipfile.ZipFile(archive) as z:
        for destination in required_features:
            if destination.exists():continue
            names=[name for name in z.namelist() if Path(name).name==destination.name]
            if len(names)!=1:raise ValueError('Feature split thiếu/trùng trong archive: '+destination.name)
            with z.open(names[0]) as src,destination.open('wb') as dst:shutil.copyfileobj(src,dst)
assert all(p.is_file() for p in required_features)
print('Feature đủ cả TRAIN / VAL / TEST:',[p.name for p in required_features])


In [ ]:
from pathlib import Path
SAVE_DIR=str(Path(CONFIG['save_root'])/CONFIG['run_id'])
Path(SAVE_DIR).mkdir(parents=True,exist_ok=True)
print('SAVE_DIR =',SAVE_DIR)


In [ ]:
# 8) Patch CHỈ evaluation/logging cho W&B
# Reset source official trước khi patch -> chạy lại cell cũng không bị patch chồng.

from pathlib import Path
import subprocess
import py_compile

ROOT = Path(CONFIG["repo_dir"])
MAIN = ROOT / "code/GroundFormer/main_dual.py"
TRAIN = ROOT / "code/GroundFormer/train/trainval_gdqa.py"
EVAL_GROUND = ROOT / "code/GroundFormer/eval_ground.py"

subprocess.check_call([
    "git", "-C", str(ROOT), "checkout", "--",
    "code/GroundFormer/main_dual.py",
    "code/GroundFormer/train/trainval_gdqa.py",
    "code/GroundFormer/eval_ground.py",
])
print("✅ Reset source về official")

# A) eval_ground.py -> return metrics
s = EVAL_GROUND.read_text()

needle = "\ndef _eval_groups"
if needle not in s:
    raise RuntimeError("Không tìm thấy _eval_groups")

return_block = """
    return {
        "Acc@QA": acc_qa,
        "Acc@GQA": acc_gqa,
        "mIoP": mIoP,
        "IoP@0.3": crtp3 * 1.0 / cnt * 100,
        "IoP@0.5": crtp5 * 1.0 / cnt * 100,
        "mIoU": mIoU,
        "IoU@0.3": crt3 * 1.0 / cnt * 100,
        "IoU@0.5": crt5 * 1.0 / cnt * 100,
    }

"""

pos = s.index(needle)
s = s[:pos] + "\n" + return_block + s[pos + 1:]

old_main = (
    "    eval_ground(gt_ground, pred_ground, pred_qa, subset=None)\n"
    "    eval_by_type_fn(gt_ground, pred_ground, pred_qa, sample_list, subset=None)\n"
)

new_main = (
    "    metrics = eval_ground(gt_ground, pred_ground, pred_qa, subset=None)\n"
    "    eval_by_type_fn(gt_ground, pred_ground, pred_qa, sample_list, subset=None)\n"
    "    return metrics\n"
)

if old_main not in s:
    raise RuntimeError("Không tìm thấy eval_ground.main() official block")

s = s.replace(old_main, new_main, 1)
EVAL_GROUND.write_text(s)
print("✅ eval_ground.py return metrics")

# B) trainval_gdqa.py -> train history trong epoch
s = TRAIN.read_text()

if "import wandb\n" not in s:
    s = s.replace("import logging\n", "import logging\nimport wandb\n", 1)

old_train_log = """        if (i + 1) % (len(train_loader) // args.freq_display) == 0:
            logging.info(f"Epoch {epoch + 1}/{args.epochs}, Train acc: {running_acc.avg:.2%}, "
                         f"QA loss: {running_a_loss.avg:.4f}")
            running_acc.reset()
            running_a_loss.reset()
"""

# Fallback exact official indentation if the string above does not match after dedent handling.
if old_train_log not in s:
    old_train_log = (
        "        if (i + 1) % (len(train_loader) // args.freq_display) == 0:\n"
        "            logging.info(f\"Epoch {epoch + 1}/{args.epochs}, Train acc: {running_acc.avg:.2%}, \"\n"
        "                         f\"QA loss: {running_a_loss.avg:.4f}\")\n"
        "            running_acc.reset()\n"
        "            running_a_loss.reset()\n"
    )

new_train_log = (
    "        if (i + 1) % (len(train_loader) // args.freq_display) == 0:\n"
    "            logging.info(\n"
    "                f\"Epoch {epoch + 1}/{args.epochs}, \"\n"
    "                f\"Train acc: {running_acc.avg:.2%}, \"\n"
    "                f\"QA loss: {running_a_loss.avg:.4f}\"\n"
    "            )\n"
    "\n"
    "            if wandb.run is not None:\n"
    "                train_step = epoch * len(train_loader) + (i + 1)\n"
    "                wandb.log({\n"
    "                    \"train_step\": train_step,\n"
    "                    \"epoch\": epoch + 1,\n"
    "                    \"train/accuracy\": 100.0 * running_acc.avg,\n"
    "                    \"train/qa_loss\": running_a_loss.avg,\n"
    "                    \"train/lr\": optimizer.param_groups[0][\"lr\"],\n"
    "                })\n"
    "\n"
    "            running_acc.reset()\n"
    "            running_a_loss.reset()\n"
)

if old_train_log not in s:
    raise RuntimeError("Không tìm thấy official train logging block")

s = s.replace(old_train_log, new_train_log, 1)
TRAIN.write_text(s)
print("✅ train/* update theo train_step")

# C) main_dual.py -> W&B init + VAL every epoch + TEST final
s = MAIN.read_text()

if "import wandb\n" not in s:
    s = s.replace("import time\n", "import time\nimport wandb\n", 1)

init_target = "        logging.info(args)\n"

init_replacement = (
    "        logging.info(args)\n"
    "\n"
    "        wandb.init(\n"
    "            project=os.environ.get(\"WANDB_PROJECT\", \"GroundFormer\"),\n"
    "            name=os.environ.get(\"WANDB_RUN_NAME\", \"nextgqa_bs16_official_15ep\"),\n"
    "            config=vars(args),\n"
    "        )\n"
    "\n"
    "        wandb.define_metric(\"train_step\")\n"
    "        wandb.define_metric(\"train/*\", step_metric=\"train_step\")\n"
    "\n"
    "        wandb.define_metric(\"epoch\")\n"
    "        wandb.define_metric(\"val/*\", step_metric=\"epoch\")\n"
    "        wandb.define_metric(\"time/*\", step_metric=\"epoch\")\n"
)

if init_target not in s:
    raise RuntimeError("Không tìm thấy W&B init target")

s = s.replace(init_target, init_replacement, 1)

start_marker = (
    "            val_acc, results, ground = "
    "eval(model, val_loader, a2v, args, test=False)\n"
)
end_marker = "            if args.save_ep:\n"

if start_marker not in s or end_marker not in s:
    raise RuntimeError("Không tìm thấy validation region")

start = s.index(start_marker)
end = s.index(end_marker, start)

new_val_region = """            val_acc, results, ground = eval(model, val_loader, a2v, args, test=False)
            _t_eval = time.time() - _t0

            if rank == 0:
                logging.info(
                    f"[Time] Epoch {epoch+1}/{args.epochs}  "
                    f"train={_t_train:.1f}s  "
                    f"eval={_t_eval:.1f}s  "
                    f"total={_t_train+_t_eval:.1f}s"
                )

                save_to(osp.join(args.save_dir, "val-res.json"), results)
                save_to(osp.join(args.save_dir, "val_ground_att.json"), ground)

                anno_dir = args.load_path

                getspan.main(
                    args.save_dir,
                    "val",
                    anno_dir=anno_dir
                )

                val_gm = eval_ground.main(
                    args.save_dir,
                    "val_ground_ada.json",
                    data_dir=anno_dir,
                    dataset=args.dataset
                )

                if wandb.run is not None:
                    wandb.log({
                        "epoch": epoch + 1,
                        "val/accuracy": 100.0 * val_acc,
                        "val/Acc@QA": val_gm["Acc@QA"],
                        "val/Acc@GQA": val_gm["Acc@GQA"],
                        "val/mIoP": val_gm["mIoP"],
                        "val/IoP@0.3": val_gm["IoP@0.3"],
                        "val/IoP@0.5": val_gm["IoP@0.5"],
                        "val/mIoU": val_gm["mIoU"],
                        "val/IoU@0.3": val_gm["IoU@0.3"],
                        "val/IoU@0.5": val_gm["IoU@0.5"],
                        "time/train_sec": _t_train,
                        "time/eval_sec": _t_eval,
                        "time/epoch_sec": _t_train + _t_eval,
                    })

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch

                torch.save(
                    model.module.state_dict(),
                    os.path.join(args.save_dir, "best_model.pth")
                )

                if rank == 0:
                    import shutil

                    shutil.copyfile(
                        osp.join(args.save_dir, "val-res.json"),
                        osp.join(args.save_dir, "best_val-res.json")
                    )
                    shutil.copyfile(
                        osp.join(args.save_dir, "val_ground_att.json"),
                        osp.join(args.save_dir, "best_val_ground_att.json")
                    )
                    shutil.copyfile(
                        osp.join(args.save_dir, "val_ground_ada.json"),
                        osp.join(args.save_dir, "best_val_ground_ada.json")
                    )

                    if wandb.run is not None:
                        wandb.run.summary["best_epoch"] = epoch + 1
                        wandb.run.summary["best_val_accuracy"] = 100.0 * val_acc

                        for k, v in val_gm.items():
                            wandb.run.summary[f"best_val/{k}"] = v

"""

# Normalize indentation of the replacement block to exactly 12 spaces at the loop body.
import textwrap as _tw
new_val_region = _tw.dedent(new_val_region)
new_val_region = "\n".join(
    ("            " + line if line else "")
    for line in new_val_region.splitlines()
) + "\n"

s = s[:start] + new_val_region + s[end:]

final_target = (
    "    eval_ground.main(args.save_dir, "
    "f'{args.test}_ground_ada.json', "
    "data_dir=anno_dir, dataset=args.dataset)\n"
)

final_replacement = (
    "    test_gm = eval_ground.main(\n"
    "        args.save_dir,\n"
    "        f'{args.test}_ground_ada.json',\n"
    "        data_dir=anno_dir,\n"
    "        dataset=args.dataset\n"
    "    )\n"
    "\n"
    "    if rank == 0 and wandb.run is not None:\n"
    "        test_metrics = {\n"
    "            \"test/accuracy\": 100.0 * test_acc,\n"
    "            \"test/Acc@QA\": test_gm[\"Acc@QA\"],\n"
    "            \"test/Acc@GQA\": test_gm[\"Acc@GQA\"],\n"
    "            \"test/mIoP\": test_gm[\"mIoP\"],\n"
    "            \"test/IoP@0.3\": test_gm[\"IoP@0.3\"],\n"
    "            \"test/IoP@0.5\": test_gm[\"IoP@0.5\"],\n"
    "            \"test/mIoU\": test_gm[\"mIoU\"],\n"
    "            \"test/IoU@0.3\": test_gm[\"IoU@0.3\"],\n"
    "            \"test/IoU@0.5\": test_gm[\"IoU@0.5\"],\n"
    "        }\n"
    "\n"
    "        wandb.log(test_metrics)\n"
    "        for k, v in test_metrics.items():\n"
    "            wandb.run.summary[k] = v\n"
    "        wandb.finish()\n"
)

if final_target not in s:
    raise RuntimeError("Không tìm thấy final test eval block")

s = s.replace(final_target, final_replacement, 1)
MAIN.write_text(s)

for p in [MAIN, TRAIN, EVAL_GROUND]:
    py_compile.compile(str(p), doraise=True)
    print("✅ compile:", p.relative_to(ROOT))

main_check = MAIN.read_text()
train_check = TRAIN.read_text()
eval_check = EVAL_GROUND.read_text()

assert '"val/Acc@GQA"' in main_check
assert '"val/mIoP"' in main_check
assert '"test/Acc@GQA"' in main_check
assert '"train/accuracy"' in train_check
assert "return metrics" in eval_check

print("\n✅ FULL W&B patch OK")
print("train/* -> trong epoch")
print("val/*   -> mỗi epoch")
print("test/*  -> cuối cùng trên best_model.pth")

# Every-epoch observation support: outside original model/loss code.
import base64, hashlib, importlib.util
CORE_B64='...'
CORE_SHA256='d606bfff3f98d4c73907ddaca112508622592cb2ac411f7c56be75f40ffc6065'
PATCH_B64='IiIiQWRkIG9ic2VydmF0aW9uIHRvIG9mZmljaWFsIGxvZ2dpbmctcGF0Y2hlZCBtYWluOyBvcHRpbWl6ZXIvbW9kZWwgc291cmNlIHVuY2hhbmdlZC4iIiIKaW1wb3J0IGFzdAoKZGVmIHBhdGNoX21haW4oc291cmNlKToKICAgIGluc2VydD0nJydmcm9tIGdmX2Vwb2NoX2FsbF9zcGxpdHMgaW1wb3J0IEVwb2NoT2JzZXJ2ZXIsIHByZXNlcnZlX3JuZwpmcm9tIGRhdGFsb2FkZXIudnFhX2xvYWRlciBpbXBvcnQgVmlkZW9RQURhdGFzZXQsIHZpZGVvcWFfY29sbGF0ZV9mbgpmcm9tIHRvb2xzLmdldHNwYW4gaW1wb3J0IGZpbmRfc2VnX2FkYQonJycKICAgIHNvdXJjZT1pbnNlcnQrc291cmNlCiAgICBzb3VyY2U9c291cmNlLnJlcGxhY2UoJ3dhbmRiLmRlZmluZV9tZXRyaWMoInRyYWluLyoiLCBzdGVwX21ldHJpYz0idHJhaW5fc3RlcCIpJywnd2FuZGIuZGVmaW5lX21ldHJpYygib3B0aW0vKiIsIHN0ZXBfbWV0cmljPSJ0cmFpbl9zdGVwIiknKQogICAgdGFyZ2V0PSIgICAgaWYgYXJncy50ZXN0ID09ICd0cmFpbic6IgogICAgaWYgc291cmNlLmNvdW50KHRhcmdldCkhPTE6cmFpc2UgVmFsdWVFcnJvcignVW5rbm93biBvZmZpY2lhbCB0cmFpbmluZyBicmFuY2gnKQogICAgaW5pdGlhbGl6YXRpb249JycnICAgIHdpdGggcHJlc2VydmVfcm5nKCk6CiAgICAgICAgZXBvY2hfb2JzZXJ2ZXIgPSBFcG9jaE9ic2VydmVyKGFyZ3Muc2F2ZV9kaXIsIGFyZ3MubG9hZF9wYXRoLCBhcmdzLmZlYXR1cmVzX3BhdGgsCiAgICAgICAgICAgIGJlcnRfdG9rZW5pemVyLCBWaWRlb1FBRGF0YXNldCwgdmlkZW9xYV9jb2xsYXRlX2ZuLCBmaW5kX3NlZ19hZGEsCiAgICAgICAgICAgIHJ1bj13YW5kYi5ydW4sIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplX3ZhbCwgd29ya2Vycz1hcmdzLm51bV90aHJlYWRfcmVhZGVyKQonJycKICAgIHNvdXJjZT1zb3VyY2UucmVwbGFjZSh0YXJnZXQsaW5pdGlhbGl6YXRpb24rdGFyZ2V0LDEpCiAgICByb290PWFzdC5wYXJzZShzb3VyY2UpCiAgICBsb29wPW5leHQobiBmb3IgbiBpbiBhc3Qud2Fsayhyb290KSBpZiBpc2luc3RhbmNlKG4sYXN0LkZvcikgYW5kIGFzdC51bnBhcnNlKG4uaXRlcik9PSdyYW5nZShhcmdzLmVwb2NocyknKQogICAgY2FuZGlkYXRlcz1bXQogICAgZm9yIG4gaW4gYXN0LndhbGsobG9vcCk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShuLGFzdC5JZikgYW5kIGFzdC51bnBhcnNlKG4udGVzdCk9PSd3YW5kYi5ydW4gaXMgbm90IE5vbmUnOgogICAgICAgICAgICBpZiBhbnkoaXNpbnN0YW5jZSh4LGFzdC5Db25zdGFudCkgYW5kIHgudmFsdWU9PSd2YWwvYWNjdXJhY3knIGZvciB4IGluIGFzdC53YWxrKG4pKToKICAgICAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKG4pCiAgICBpZiBsZW4oY2FuZGlkYXRlcykhPTE6cmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgb25lIG9mZmljaWFsIHZhbCBsb2dnaW5nIGJsb2NrJykKICAgIG9sZD1jYW5kaWRhdGVzWzBdO2xpbmVzPXNvdXJjZS5zcGxpdGxpbmVzKFRydWUpCiAgICByZXBsYWNlPScnJyAgICAgICAgICAgICAgICBlcG9jaF9vYnNlcnZlci5lcG9jaChtb2RlbCwgZXBvY2ggKyAxLAogICAgICAgICAgICAgICAgICAgIGVwb2NoX29ic2VydmVyLnJlY29yZHNfZnJvbV9vdXRwdXRzKCd2YWwnLCByZXN1bHRzLCBncm91bmQpLAogICAgICAgICAgICAgICAgICAgIHsnbHInOiBvcHRpbWl6ZXIucGFyYW1fZ3JvdXBzWzBdWydsciddLCAndHJhaW5fc2Vjb25kcyc6IF90X3RyYWluLAogICAgICAgICAgICAgICAgICAgICAndmFsX3NlY29uZHMnOiBfdF9ldmFsfSkKJycnCiAgICBsaW5lc1tvbGQubGluZW5vLTE6b2xkLmVuZF9saW5lbm9dPVtyZXBsYWNlXTtzb3VyY2U9Jycuam9pbihsaW5lcykKICAgIG1hcmtlcj0nICAgICAgICBsb2dnaW5nLmluZm8oZiJCZXN0IHZhbCBtb2RlbCBhdCBlcG9jaCB7YmVzdF9lcG9jaCArIDF9IiknCiAgICBpZiBzb3VyY2UuY291bnQobWFya2VyKSE9MTpyYWlzZSBWYWx1ZUVycm9yKCdVbmtub3duIGJlc3QgdmFsaWRhdGlvbiBtYXJrZXInKQogICAgc291cmNlPXNvdXJjZS5yZXBsYWNlKG1hcmtlcixtYXJrZXIrJ1xuICAgICAgICBlcG9jaF9vYnNlcnZlci5zZWxlY3RlZChiZXN0X2Vwb2NoICsgMSknLDEpCiAgICBmaW5hbF9tYXJrZXI9JyAgICBpZiByYW5rID09IDAgYW5kIHdhbmRiLnJ1biBpcyBub3QgTm9uZTpcbiAgICAgICAgdGVzdF9tZXRyaWNzID0geycKICAgIGlmIHNvdXJjZS5jb3VudChmaW5hbF9tYXJrZXIpIT0xOnJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gZmluYWwtdGVzdCBsb2dnaW5nIGJsb2NrJykKICAgIHBvc2l0aW9uPXNvdXJjZS5pbmRleChmaW5hbF9tYXJrZXIpCiAgICBzb3VyY2U9c291cmNlWzpwb3NpdGlvbl0rc291cmNlW3Bvc2l0aW9uOl0ucmVwbGFjZSgnInRlc3QvJywnInNlbGVjdGVkX3Rlc3QvJykKICAgIGNvbXBpbGUoc291cmNlLCdvZmZpY2lhbF9lbmRfZXBvY2hfb2JzZXJ2YXRpb25zJywnZXhlYycpCiAgICByZXR1cm4gc291cmNlCg=='
PATCH_SHA256='0e38a5f8c86fa309b63d219415c1bab42a6b2ef1214521b84d9ccdc2cf1516f3'
core_bytes=base64.b64decode(CORE_B64);assert hashlib.sha256(core_bytes).hexdigest()==CORE_SHA256
patch_bytes=base64.b64decode(PATCH_B64);assert hashlib.sha256(patch_bytes).hexdigest()==PATCH_SHA256
SUPPORT=ROOT/'code/GroundFormer'
(SUPPORT/'gf_epoch_all_splits.py').write_bytes(core_bytes)
patch_path=SUPPORT/'gf_original_epoch_patch.py';patch_path.write_bytes(patch_bytes)
patch_spec=importlib.util.spec_from_file_location('gf_original_epoch_patch',patch_path)
epoch_patch=importlib.util.module_from_spec(patch_spec);patch_spec.loader.exec_module(epoch_patch)
MAIN.write_text(epoch_patch.patch_main(MAIN.read_text()),encoding='utf-8')
TRAIN.write_text(TRAIN.read_text().replace('"train/','"optim/'),encoding='utf-8')
for p in [MAIN,TRAIN,SUPPORT/'gf_epoch_all_splits.py']:py_compile.compile(str(p),doraise=True)
print('Đã bật TRAIN / VAL / TEST cuối mọi epoch; optim/* chỉ là log minibatch.')


In [ ]:
# 9) Tạo script 15 epochs từ official script
# Chỉ đổi epochs và thêm freq_display=20 (logging-only).

from pathlib import Path

ROOT = Path(CONFIG["repo_dir"])
official_script = ROOT / "scripts/train_nextgqa.sh"
run_script = ROOT / "scripts/train_nextgqa_15ep_fullwandb.sh"

lines = official_script.read_text().splitlines()

epoch_idx = [
    i for i, line in enumerate(lines)
    if line.strip().startswith("--epochs=")
]

if len(epoch_idx) != 1:
    raise RuntimeError(
        f"Official script phải có đúng 1 dòng --epochs=, hiện có {len(epoch_idx)}"
    )

i = epoch_idx[0]
indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
has_slash = lines[i].rstrip().endswith("\\")
lines[i] = f"{indent}--epochs=15" + (" \\" if has_slash else "")

freq_idx = [
    i for i, line in enumerate(lines)
    if line.strip().startswith("--freq_display=")
]

if freq_idx:
    if len(freq_idx) != 1:
        raise RuntimeError("Có nhiều dòng --freq_display=")
    i = freq_idx[0]
    indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
    has_slash = lines[i].rstrip().endswith("\\")
    lines[i] = f"{indent}--freq_display=20" + (" \\" if has_slash else "")
else:
    save_idx = next(
        (
            i for i, line in enumerate(lines)
            if line.strip().startswith("--save_dir=")
        ),
        None,
    )
    if save_idx is None:
        raise RuntimeError("Không tìm thấy --save_dir=")

    lines.insert(save_idx, "  --freq_display=20 \\")

save_idx = next(i for i,line in enumerate(lines) if line.strip().startswith("--save_dir="))
lines.insert(save_idx, "  --eval_split=test \\")
patched_text = "\n".join(lines) + "\n"
run_script.write_text(patched_text)
run_script.chmod(0o755)

print("=== CONFIG CHECK ===")

for key in [
    "--epochs",
    "--lr",
    "--batch_size",
    "--batch_size_val",
    "--sigma",
    "--seed",
    "--vg_loss",
    "--freq_display",
]:
    line = next(
        (
            x.strip()
            for x in lines
            if x.strip().startswith(key + "=")
        ),
        None,
    )
    print(line)

def get_value(key):
    line = next(
        x.strip()
        for x in lines
        if x.strip().startswith(key + "=")
    )
    value = line.split("=", 1)[1].strip()
    if value.endswith("\\"):
        value = value[:-1].strip()
    return value

assert get_value("--epochs") == "15"
assert get_value("--batch_size") == "16"
assert get_value("--sigma") == "9"
assert get_value("--freq_display") == "20"

print("\n✅ Script ready:", run_script)


In [ ]:
import os, subprocess, json
from pathlib import Path
os.environ['WANDB_PROJECT']=CONFIG['wandb_project']
os.environ['WANDB_RUN_NAME']=CONFIG['run_id']
run_script=Path(CONFIG['repo_dir'])/'scripts/train_nextgqa_15ep_fullwandb.sh'
complete=Path(SAVE_DIR)/'COMPLETE_TRAIN_VAL_TEST.json'
if complete.exists():
    import hashlib
    saved=json.loads(complete.read_text())
    if saved['source_commit']!=PINNED_REV:raise ValueError('Completed run source differs')
    for name,expected in saved['output_sha256'].items():
        p=Path(SAVE_DIR)/name
        h=hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
        if h.hexdigest()!=expected:raise ValueError('Completed artifact changed: '+name)
    print('Train/val/test đã hoàn tất, hash khớp; không train lại.')
else:
    if (Path(SAVE_DIR)/'best_model.pth').exists():
        raise RuntimeError('SAVE_DIR có run chưa được khóa hoàn tất. Đổi run_id để train mới; không ghi đè historical weights.')
    cmd=['bash',str(run_script),'0',SAVE_DIR]
    print('TRAIN 15 epoch -> chọn best QA VAL -> TEST:',cmd)
    process=subprocess.Popen(cmd,cwd=CONFIG['repo_dir'],stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (Path(SAVE_DIR)/'NOTEBOOK_STDOUT.log').open('w',encoding='utf-8') as log:
        for line in process.stdout:print(line,end='');log.write(line);log.flush()
    process.wait()
    if process.returncode:raise RuntimeError('Train/val/test chưa hoàn tất; xem traceback trong log.')


In [ ]:
# 11) Kiểm tra VAL + BEST VAL + TEST outputs
from pathlib import Path

save_dir = Path(SAVE_DIR)

expected = [
    save_dir / "best_model.pth",
    save_dir / "val-res.json",
    save_dir / "val_ground_att.json",
    save_dir / "val_ground_ada.json",
    save_dir / "best_val-res.json",
    save_dir / "best_val_ground_att.json",
    save_dir / "best_val_ground_ada.json",
    save_dir / "test-res.json",
    save_dir / "test_ground_att.json",
    save_dir / "test_ground_ada.json",
    save_dir / "stdout.log",
]

missing = [p for p in expected if not p.exists()]

print("=== OUTPUT CHECK ===")
for p in expected:
    print(("✅" if p.exists() else "❌"), p.name)

if missing:
    raise RuntimeError(
        "Thiếu output:\n"
        + "\n".join(str(p) for p in missing)
    )

print("\n✅ Đủ outputs")

# Independent population check and saved TEST metrics, even when W&B is offline.
import csv, hashlib, importlib.util, json, re, sys, zipfile
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
with (ANNO_DIR/'test.csv').open(newline='',encoding='utf-8-sig') as f:official_rows=list(csv.DictReader(f))
expected_ids={r['video_id']+'_'+r['qid'] for r in official_rows}
qa=json.loads((save_dir/'test-res.json').read_text());spans=json.loads((save_dir/'test_ground_ada.json').read_text())
assert len(expected_ids)==5553 and set(qa)==set(spans)==expected_ids
assert len({r['video_id'] for r in official_rows})==990
sys.path.insert(0,str(ROOT/'code/GroundFormer'))
spec=importlib.util.spec_from_file_location('gf_original_test_metrics',ROOT/'code/GroundFormer/eval_ground.py')
ev=importlib.util.module_from_spec(spec);spec.loader.exec_module(ev)
test_metrics=ev.main(SAVE_DIR,'test_ground_ada.json',data_dir=str(ANNO_DIR),dataset='nextgqa')
test_metrics.update(split='test',questions=5553,videos=990,checkpoint_selection='highest validation QA')
(save_dir/'TEST_METRICS.json').write_text(json.dumps(test_metrics,indent=2),encoding='utf-8')
logs=(save_dir/'stdout.log').read_text(errors='replace')
epochs=re.findall(r'Best val model at epoch (\d+)',logs)
if not epochs:raise ValueError('No verified best-validation epoch in training log')
selection={'source_commit':PINNED_REV,'epochs':15,'selected_epoch':int(epochs[-1]),
           'checkpoint':'best_model.pth','criterion':'highest validation QA, strict >, earlier tie retained',
           'test_evaluations':'one final selected checkpoint; test never chooses weights',
           'test_annotations_sha256':{p.name:file_sha(p) for p in [ANNO_DIR/'test.csv',ANNO_DIR/'gsub_test.json',ANNO_DIR/'frame2time_test.json']}}
(save_dir/'CHECKPOINT_SELECTION.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
checks={p.name:file_sha(p) for p in expected+[save_dir/'TEST_METRICS.json',save_dir/'CHECKPOINT_SELECTION.json']}
completion=dict(selection,status='EXECUTED_TRAIN_VAL_TEST',output_sha256=checks)
(save_dir/'COMPLETE_TRAIN_VAL_TEST.json').write_text(json.dumps(completion,indent=2),encoding='utf-8')
print('Selected best-val QA epoch:',selection['selected_epoch'])
print('TEST metrics:',json.dumps(test_metrics,indent=2))
# Require all three full split observations at all 15 epoch boundaries.
epoch_hashes={}
for epoch in range(1,16):
    p=save_dir/'EPOCH_OBSERVATIONS'/f'epoch_{epoch:02d}'/'ALL_SPLITS.json'
    values=json.loads(p.read_text())
    assert values['epoch']==epoch
    assert all(values['splits'][s]['questions']==n for s,n in [('train',34132),('val',3358),('test',5553)])
    assert values['splits']['train']['GQA'] is None
    epoch_hashes[p.relative_to(save_dir).as_posix()]=file_sha(p)
completion['output_sha256'].update(epoch_hashes)
completion['evaluation_frequency']='full train/val/test at end of EVERY epoch'
(save_dir/'COMPLETE_TRAIN_VAL_TEST.json').write_text(json.dumps(completion,indent=2),encoding='utf-8')
print('Đủ 45 lượt chấm split: 15 epoch × train/val/test.')
archive=save_dir.parent/(save_dir.name+'_RESULTS.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in save_dir.rglob('*'):
        if p.is_file() and 'wandb' not in p.relative_to(save_dir).parts and p.suffix in ['.json','.log','.csv','.md','.gz','.npz']:z.write(p,p.relative_to(save_dir).as_posix())
print('ZIP:',archive)
if CONFIG['download_zip']:
    from google.colab import files
    files.download(str(archive))


W&B: `train/*`, `val/*`, `test/*` theo trục epoch; `optim/*` là loss/accuracy minibatch. `selected_test/*` chỉ báo checkpoint best QA validation. `EPOCH_OBSERVATIONS/epoch_XX/ALL_SPLITS.json` lưu số chính xác, có prediction và attention val/test để audit. IoP/IoU/GQA train = N/A vì không có GT thời gian train.